In [1]:
# Save mortality at each grid point using central estimates only

In [2]:
import os
import xarray as xr
import numpy as np
from utils.mortality_utils import att_frac
from utils.mortality_utils import mortality
from utils.utils import get_scenario_config

In [3]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"

In [4]:
# === Load data ===
bmr_file = "GBD_BMR_Country_Mask_COPD_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path).sel(quantile="mean")  # central estimate

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(BMR, method="nearest", tolerance=1e-9)

In [5]:
# === Calculate beta for RR GBD 2021 ===

# Relative risk for 10ppb increase in OSDMA8, GBD 2021
RR_10ppb = 1.074  # [95% CI 1.014 – 1.137]
# equation is: RR = e^(beta*(x-TMREL)) where RR_10ppb = e^(10beta)
beta = np.log(RR_10ppb)/10

# TMREL from GBD 2021
TMREL = 32.4  # [95% Uniform CI 29.1 – 35.7]

In [7]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "UKESM1"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

O3_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/gridpoint_mortality/"

# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")
    # {years.stop - 1} from OSDMA8 calculation
    dates = f"{years.start}-{years.stop - 1}"

    o3_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    o3_path = os.path.join(O3_DIR, o3_file)
    o3 = xr.open_dataarray(o3_path)

    # Adjust indices to match (with small tolerance)
    # e.g., max 1e-7 km distance
    o3 = o3.reindex_like(BMR, method="nearest", tolerance=1e-9, fill_value=0)

    # Flag if any nans present (i.e. reindex was out of tolerance distance)
    assert not population.isnull().any()

    M = []

    for year in years:
        print(f"Processing year {year}")
        POP = pop.sel(year=year)
        AF = att_frac(o3.sel(year=year), TMREL, beta)
        mortality_year = mortality(AF, BMR, POP)
        M.append(mortality_year)

    M_cleaned = [da.drop_vars("year", errors="ignore") for da in M]
    mortality_timeseries = xr.concat(
        M_cleaned,
        dim=(xr.DataArray(o3["year"].values,
                          dims="year", name="year")))

    description = ("Total COPD mortality due to surface ozone using "
                   "central estimates only - scripts "
                   "by A.F. Wells (2025)")
    mortality_timeseries.attrs["description"] = description
    mortality_timeseries.attrs["ensemble_number"] = ens_num
    mortality_timeseries.attrs["scenario"] = scenario
    mortality_timeseries.attrs["model"] = model

    out_file = f"Mortality_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving mortality timeseries to {out_path}")
    mortality_timeseries.to_netcdf(out_path)

print("All processing complete.")

Processing SSP245_G6, Ensemble 01
Processing year 2020
Processing year 2021
Processing year 2022
Processing year 2023
Processing year 2024
Processing year 2025
Processing year 2026
Processing year 2027
Processing year 2028
Processing year 2029
Processing year 2030
Processing year 2031
Processing year 2032
Processing year 2033
Processing year 2034
Processing year 2035
Processing year 2036
Processing year 2037
Processing year 2038
Processing year 2039
Processing year 2040
Processing year 2041
Processing year 2042
Processing year 2043
Processing year 2044
Processing year 2045
Processing year 2046
Processing year 2047
Processing year 2048
Processing year 2049
Processing year 2050
Processing year 2051
Processing year 2052
Processing year 2053
Processing year 2054
Processing year 2055
Processing year 2056
Processing year 2057
Processing year 2058
Processing year 2059
Processing year 2060
Processing year 2061
Processing year 2062
Processing year 2063
Processing year 2064
Processing year 2065
